# 03 – Classify for RiverEggCode
**Purpose:** ....


**Workflow:**

3.0 | Imports

3.1 | Configuration cell

3.2 | 

&nbsp;&nbsp;&nbsp;&nbsp;3.2.1 | Classify Reaches

&nbsp;&nbsp;&nbsp;&nbsp;3.2.2 | Aggregate Reaches into Eggs


---
## 3.0 | Imports

In [1]:
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import sys
import importlib


sys.path.append(os.path.join("..", "src"))
from _0_config_paths import DATA_PROCESSED, DATA_OUTPUT
from _3_2_1_classifier import classify_all
from _3_2_2_eggcode   import build_all_eggs, egg_to_dataframe, egg_to_string, build_basin6_eggs

---
## 3.1 | Configuration cell

In [2]:
STUDY_AREA = "naryn"

# Input: joined dataset from Notebook 02 
IN_JOINED = os.path.join(DATA_PROCESSED, f"sword_{STUDY_AREA}_joined_final.gpkg")

# Outputs
OUT_CLASSIFIED = os.path.join(DATA_PROCESSED, f"sword_{STUDY_AREA}_classified.gpkg")
OUT_EGGS_CSV   = os.path.join(DATA_OUTPUT, f"{STUDY_AREA}_eggs.csv")

print("Configuration set:")
print(f"Input : {IN_JOINED}")
print(f"Output : {OUT_CLASSIFIED}")
print(f"Eggs : {OUT_EGGS_CSV}")

Configuration set:
Input : C:\Users\Duck\Documents\Studium\EAGLE\04_semester\0_InnoLab\RiverEggCode\data\processed\sword_naryn_joined_final.gpkg
Output : C:\Users\Duck\Documents\Studium\EAGLE\04_semester\0_InnoLab\RiverEggCode\data\processed\sword_naryn_classified.gpkg
Eggs : C:\Users\Duck\Documents\Studium\EAGLE\04_semester\0_InnoLab\RiverEggCode\data\outputs\naryn_eggs.csv


---
## 3.2 |
### 3.2.1 | Classify Reaches

Load joined dataset from notebook 02 and classify each SWORD reach individualy. Every SWORD reach gets its own egg_ columns based on its own measured values (slope, n_chan_mod, facc etc.):

In [3]:
gdf = gpd.read_file(IN_JOINED)

print(f"Reaches loaded: {len(gdf)}")
print(f"CRS : {gdf.crs}")


Reaches loaded: 140
CRS : EPSG:4326


In [4]:
# Explore SWORD reach_id structure for egg grouping
# reach_id format: CBBBBBRRRRT
# First 6 digits = Pfafstetter Level 6 basin (potential egg group)
print("\nBasin Level 6 distribution (first 6 digits of reach_id):")
print(gdf["reach_id"].astype(str).str[:6].value_counts())
print(f"\nUnique basins: {gdf['reach_id'].astype(str).str[:6].nunique()}")


Basin Level 6 distribution (first 6 digits of reach_id):
reach_id
462191    38
462199    32
462192    17
462193    12
462196    11
462197    10
462198     9
462195     7
462194     4
Name: count, dtype: int64

Unique basins: 9


In [5]:
gdf_classified = classify_all(gdf)

# Show result
egg_cols = [c for c in gdf_classified.columns if c.startswith("egg_")]
print("\nClassified columns:", egg_cols)
gdf_classified[["reach_id", "dist_out"] + egg_cols].head(10)

# Save classified reaches (one row per SWORD reach, with egg_ columns added):
os.makedirs(DATA_PROCESSED, exist_ok=True)
gdf_classified.to_file(OUT_CLASSIFIED, driver="GPKG")
print(f"Saved: {OUT_CLASSIFIED}")

 egg_SL: Done
 egg_PL: Done
 egg_QT: Done

Classified columns: ['egg_SL', 'egg_PL', 'egg_QT']
Saved: C:\Users\Duck\Documents\Studium\EAGLE\04_semester\0_InnoLab\RiverEggCode\data\processed\sword_naryn_classified.gpkg


### 3.2.2 | Aggregate reaches into eggs
One Egg per global_id_GRITv1.0. Reaches sorted upstream → downstream via dist_out.

In [6]:
# eggs = build_all_eggs(
#     gdf = gdf_classified,
#     grouped_col = "HYRIV_ID_RiverATLAS",
#     strahler_col = "strahler_order_RiverATLAS"
# )

# egg_df = egg_to_dataframe(eggs)

# print(f"Total Eggs: {egg_df['global_id'].nunique()}")
# print(f"Total Reaches: {len(egg_df)}")
# print(f"\nReaches per Egg (distribution):")
# print(egg_df.groupby("global_id")["reach_id"].count().describe())
# print(f"\nSample:")
# egg_df.head(15)

# # Inspecting a single Egg:
# EGG_INDEX = 8

# print(egg_to_string(eggs[EGG_INDEX]))

In [7]:
# eggs_grit = build_all_eggs(
#     gdf          = gdf_classified,
#     grouped_col  = "global_id_GRITv1.0",
#     strahler_col = "strahler_order_RiverATLAS" 
# )
# # Inspecting a single Egg:
# EGG_INDEX = 8

# print(egg_to_string(eggs_grit[EGG_INDEX]))

In [8]:
eggs = build_all_eggs(
    gdf          = gdf_classified,
    grouped_col  = "strahler_segment_id_RiverATLAS",
    strahler_col = "strahler_order_RiverATLAS" # classical Strahler from RiverATLAS
)

egg_df = egg_to_dataframe(eggs)

# Inspecting a single Egg:
EGG_INDEX = 8

print(egg_to_string(eggs[EGG_INDEX]))


Eggs built: 25
Total reaches covered: 138
═════════════════════════════════════════════════════════════════
global_id: 571.0|strahler: 3.0  |  n_reaches: 4
RT: None
-----------------------------------------------------------------
#    reach_id     len_m    SL     PL     QT     Strahler
1    46219800081  10540    5      St     1      -    
2    46219800071  19387    3      St     1      -    
3    46219800061  12327    2      St     1      -    
4    46219800051  12389    3      St     more water! -    
═════════════════════════════════════════════════════════════════


Export Eggs as .csv:

In [9]:
os.makedirs(DATA_OUTPUT, exist_ok=True)
egg_df.to_csv(OUT_EGGS_CSV, index=False)
print(f"Saved: {OUT_EGGS_CSV}")

Saved: C:\Users\Duck\Documents\Studium\EAGLE\04_semester\0_InnoLab\RiverEggCode\data\outputs\naryn_eggs.csv


In [10]:
# Creating output directory if it doesnt exist
os.makedirs(os.path.join("..", "app", "data"), exist_ok=True)

# Export classified data for Streamlit app
gdf_classified.to_parquet(
    os.path.join("..", "app", "data", f"sword_{STUDY_AREA}_classified.parquet")
)
print("Exported for app.")

Exported for app.


### 3.2.3 | Alternative: Basin Level 6 Eggs (SWORD-native grouping)

Groups all reaches sharing the same Pfafstetter Level 6 basin
No external dataset needed – basin encoded in reach_id

In [11]:
basin_eggs = build_basin6_eggs(
    gdf          = gdf_classified,
    strahler_col = "strahler_order_RiverATLAS"
)

basin_egg_df = egg_to_dataframe(basin_eggs)

print(f"Total Basin Eggs: {basin_egg_df['global_id'].nunique()}")
print(f"Total Reaches: {len(basin_egg_df)}")
print(f"\nReaches per Basin Egg:")
print(basin_egg_df.groupby("global_id")["reach_id"].count().describe())

# Inspect a single Basin Egg:
BASIN_EGG_INDEX = 2
print(egg_to_string(basin_eggs[BASIN_EGG_INDEX]))

Unique basin_6 values: 9
basin_6
462191    38
462199    32
462192    17
462193    12
462196    11
462197    10
462198     9
462195     7
462194     4
Name: count, dtype: int64
Number of groups: 9
Basin6 Eggs built: 9
Total reaches covered: 140
Total Basin Eggs: 9
Total Reaches: 140

Reaches per Basin Egg:
count     9.000000
mean     15.555556
std      11.673807
min       4.000000
25%       9.000000
50%      11.000000
75%      17.000000
max      38.000000
Name: reach_id, dtype: float64
═════════════════════════════════════════════════════════════════
global_id: 462193|n_reaches: 12
RT: None
-----------------------------------------------------------------
#    reach_id     len_m    SL     PL     QT     Strahler
1    46219300091  14231    2      St     3      5.0  
2    46219300081  11518    2      St     3      5.0  
3    46219300071  11528    2      St     3      5.0  
4    46219300061  18929    2      St     3      5.0  
5    46219300136  200      5      St     1      3.0  
6    46219

In [13]:
# Test: reload classifier module
sys.path.append(os.path.join("..", "src"))

import _3_2_1_classifier as clf
importlib.reload(clf)

# Test classify_all with default profile
gdf_classified = clf.classify_all(gdf)

# Check results
egg_cols = [c for c in gdf_classified.columns if c.startswith("egg_")]
print(f"Egg columns created: {egg_cols}")
print(f"\nSample:")
print(gdf_classified[["reach_id"] + egg_cols].head(5))

 egg_SL: Done
 egg_PL: Done
 egg_QT: Done
Egg columns created: ['egg_SL', 'egg_PL', 'egg_QT']

Sample:
      reach_id  egg_SL egg_PL       egg_QT
0  46219400056       2     St  more water!
1  46219400041       3     St  more water!
2  46219400021       3     St            3
3  46219300051       2     Br       woooow
4  46219300061       2     St            3
